In [1]:
from pathlib import Path
import pandas as pd
import random
import shutil
import yaml
from ultralytics import YOLO
import cv2
import os
import torch
import torch.utils

In [ ]:
# grab data

data_path = Path('C:/Users/sampl/Documents/code/school/aiep/aiep_project/data')

image_path = data_path / 'images'
label_path = data_path / 'labels'

processed_data_path = Path('C:/Users/sampl/Documents/code/school/aiep/aiep_project/processed_data')
train_path = processed_data_path / 'train'
train_images_path = train_path / 'images'
train_labels_path = train_path / 'labels'

val_path = processed_data_path / 'val'
val_images_path = val_path / 'images'
val_labels_path = val_path / 'labels'

# delete files
if Path.exists(processed_data_path):
    shutil.rmtree(processed_data_path)


# make files
os.mkdir(processed_data_path)
os.mkdir(train_path)
os.mkdir(train_images_path)
os.mkdir(train_labels_path)
os.mkdir(val_path)
os.mkdir(val_images_path)
os.mkdir(val_labels_path)

In [4]:
# check number of files that are being worked with

image_file_list = [path for path in Path(image_path).rglob('*')]
label_file_list = [path for path in Path(label_path).rglob('*')]

print(len(image_file_list))

95


In [5]:
# set how many training and validation

train_percent = 0.8

file_num = len(image_file_list)

train_num = int(train_percent * file_num)
val_num = file_num - train_num

print(train_num)
print(val_num)

76
19


In [6]:
# distribute data

for i, num in enumerate([train_num, val_num]):

    for index in range(num):

        img_path = random.choice(image_file_list)
        image_name = img_path.name
        base_name = img_path.stem
        text_name = base_name + '.txt'
        text_path = Path(label_path / text_name)

        # train
        if i == 0:
            shutil.copy(img_path, train_images_path)
            shutil.copy(text_path, train_labels_path)

        # val
        else:
            shutil.copy(img_path, val_images_path)
            shutil.copy(text_path, val_labels_path)

In [ ]:
# create yaml files

class_path = data_path / 'classes.txt'
data_yaml_path = processed_data_path / 'data.yaml'

data = {
    'path' : str(processed_data_path),
    'train' : 'train/images',
    'val' : 'val/images',
    'nc' : 1,
    'names' : ['Flosspick']
}

# write to yaml
with open(data_yaml_path, 'w') as file:
    yaml.dump(data, file, sort_keys = False)

In [ ]:
# load pretrained model

pretrained_path = Path(r'C:\Users\sampl\Documents\code\school\aiep\aiep_project\src\aiep_project\yolo26s.pt')

model = YOLO(pretrained_path)

# train
results = model.train(
    data = data_yaml_path,
    epochs = 60,
    imgsz = 640,
    device = 0
)

In [ ]:
# test model

model = YOLO(r'C:\Users\sampl\Documents\code\school\aiep\aiep_project\runs\detect\train\weights\best.pt')

train_results = model.predict(
    source = r'C:/Users/sampl/Documents/code/school/aiep/aiep_project/processed_data/train/images',
    conf = 0.5,
    save = True,
    save_crop = True
)